# **Book Recommendation**

---



In [ ]:
!pip install scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp311-cp311-linux_x86_64.whl size=2505171 sha256=61da4df877a16b4707751641367da31ab3ed12fd184d282d20fad5496e9c6649
  Stored in directory: /root/.cache/pip/wheels/2a/8f/6e/7e2899163e2d85d8266daab4aa1cdabec7a6c56f83c015b5af
Successfully built scikit-surprise


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
books = pd.read_csv('/content/Books.csv')
books.head(5)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [ ]:
books.isnull().sum()

,0
ISBN,0
Book-Title,0
Book-Author,0
Year-Of-Publication,0
Publisher,0
Image-URL-S,0
Image-URL-M,0
Image-URL-L,1


In [ ]:
books.duplicated().sum()

0

In [ ]:
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35566 entries, 0 to 35565
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ISBN                 35566 non-null  object
 1   Book-Title           35566 non-null  object
 2   Book-Author          35566 non-null  object
 3   Year-Of-Publication  35566 non-null  int64 
 4   Publisher            35566 non-null  object
 5   Image-URL-S          35566 non-null  object
 6   Image-URL-M          35566 non-null  object
 7   Image-URL-L          35565 non-null  object
dtypes: int64(1), object(7)
memory usage: 2.2+ MB


In [ ]:
books.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='object')

In [ ]:
books['Year-Of-Publication'] = pd.to_numeric(books['Year-Of-Publication'], errors='coerce')
books['Year-Of-Publication'].fillna(books['Year-Of-Publication'].median(), inplace=True)
books['Year-Of-Publication'] = books['Year-Of-Publication'].astype(int)
# Fill missing values with 'Unknown' (or you can drop them)
books['Book-Author'].fillna('Unknown', inplace=True)
books['Publisher'].fillna('Unknown', inplace=True)
books.drop_duplicates(inplace=True)

<ipython-input-10-384fa08ce32f>:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  books['Year-Of-Publication'].fillna(books['Year-Of-Publication'].median(), inplace=True)
<ipython-input-10-384fa08ce32f>:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col

## *Data pre-processing*

---



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate


## Content-Based Filtering

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
books['Book-Title'].fillna('', inplace=True)
tfidf_matrix = tfidf.fit_transform(books['Book-Title']) #Transforms the book titles into a numerical matrix

<ipython-input-12-d4315d34371c>:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  books['Book-Title'].fillna('', inplace=True)  # Fill missing titles with empty string


In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix) #similarlity check

In [ ]:
def recommend_books_content(book_title, num_recommendations=5):
    idx = books[books['Book-Title'] == book_title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))  #[(0, 1.0), (1, 0.85), (2, 0.30), (3, 0.50), ...]
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:num_recommendations+1]
    book_indices = [i[0] for i in sim_scores]
    return books.iloc[book_indices][['Book-Title', 'Book-Author', 'Publisher']]

In [ ]:
recommend_books_content("Nights Below Station Street", 5)


,Book-Title,Book-Author,Publisher
30875,One of These Nights,Justine Davis,Silhouette
4300,Perdido Street Station,China Mieville,Del Rey Books
27395,Ice Station,Matthew J. Reilly,St. Martin's Press
20027,Four Dark Nights,Bentley Little,Leisure Books
18549,Donovan's Station: A Novel,Robin McGrath,Creative Book Publishing


## Collaborative Filtering

In [ ]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split, cross_validate
from surprise.accuracy import rmse

# Create a dummy ratings dataset
ratings_data = {
    'User-ID': [101, 102, 103, 104, 105, 106],
    'ISBN': ['0195153448', '0002005018', '0060973129', '0374157065', '0393045218', '0195153448'],
    'Book-Rating': [8, 9, 7, 6, 10, 8]
}

ratings = pd.DataFrame(ratings_data)

# Save and reload dataset (optional)
ratings.to_csv("Ratings.csv", index=False)
ratings = pd.read_csv("Ratings.csv")  # Load dataset

#Define reader and load dataset into Surprise format
reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(ratings[['User-ID', 'ISBN', 'Book-Rating']], reader)

# Perform cross-validation to evaluate model
svd = SVD()  #Singular Value Decomposition
cross_validate(svd, data, cv=5, verbose=True)

# Train the model on the full dataset
trainset = data.build_full_trainset()
svd.fit(trainset)

# Train-test split for evaluation
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
svd.fit(trainset)

# Make predictions and evaluate performance
predictions = svd.test(testset)
rmse(predictions)  # Calculate RMSE

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8972  2.4000  0.0112  1.2000  2.4000  1.3817  0.9187  
MAE (testset)     0.7338  2.4000  0.0112  1.2000  2.4000  1.3490  0.9380  
Fit time          0.01    0.00    0.00    0.00    0.00    0.00    0.00    
Test time         0.00    0.00    0.00    0.00    0.00    0.00    0.00    
RMSE: 2.0000


2.0

In [ ]:
# Function to predict rating for a specific user & book
def predict_rating(user_id, isbn):
    pred = svd.predict(user_id, isbn)
    return pred.est  # Estimated rating

In [ ]:
#Example Usage:
predicted_rating = predict_rating(101, '0195153448')
print(f"⭐ Predicted rating for User 101 on book '0195153448': {predicted_rating:.2f}")

⭐ Predicted rating for User 101 on book '0195153448': 8.00


In [ ]:
def recommend_books_collaborative(user_id, num_recommendations=5):
    books['est_rating'] = books['ISBN'].apply(lambda x: svd.predict(user_id, x).est)
    return books.sort_values('est_rating', ascending=False)[['Book-Title', 'Book-Author', 'Publisher', 'est_rating']].head(num_recommendations)


In [ ]:
print("\n🤝 Collaborative Recommendations:")
recommend_books_collaborative(276725) # Example user ID


🤝 Collaborative Recommendations:


,Book-Title,Book-Author,Publisher,est_rating
0,Classical Mythology,Mark P. O. Morford,Oxford University Press,8.0
23712,If You Ask Me (If You Ask Me),Libby Gelman-Waxner,St Martins Pr,8.0
23706,L.A. Justice,Christopher Darden,Signet Book,8.0
23707,A Criminal Appeal,D. R. Schanker,Dell Publishing Company,8.0
23708,Blood Sport,Dick Francis,Jove Books,8.0


## Hybrid Approach

In [ ]:
def hybrid_recommendation(book_title, user_id, num_recommendations=5):
    content_recs = recommend_books_content(book_title, num_recommendations)
    collab_recs = recommend_books_collaborative(user_id, num_recommendations)
    return pd.concat([content_recs, collab_recs]).drop_duplicates().head(num_recommendations)

In [ ]:
print("\n🔥 Hybrid Recommendations:")
hybrid_recommendation("Classical Mythology", 276725)


🔥 Hybrid Recommendations:


,Book-Title,Book-Author,Publisher,est_rating
7837,Mythology,Edith Hamilton,Signet Book,NaN
20265,Illustrated Dictionary of Mythology,Philip Wilkinson,DK Publishing Inc,NaN
14293,Handbk of Greek Mythology,Herbert J. Rose,Dutton Books,NaN
30189,Bulfinch's Mythology (Laurel Classic),Thomas Bulfinch,Dell Publishing Company,NaN
20256,The Story of the World: History for the Classi...,S. Wise Bauer,Peace Hill Press,NaN


Accuracy

In [ ]:
# Import necessary libraries
from surprise import accuracy
# 💖 Calculate RMSE and MAE
rmse_score = accuracy.rmse(predictions)  # Root Mean Squared Error
mae_score = accuracy.mae(predictions)    # Mean Absolute Error
print("\n📌 Accuracy of the Hybrid Model:")
print(f"🔥 RMSE: {rmse_score:.4f}")
print(f"🔥 MAE: {mae_score:.4f}")

RMSE: 2.0000
MAE:  2.0000

📌 Accuracy of the Hybrid Model:
🔥 RMSE: 2.0000
🔥 MAE: 2.0000


In [ ]:
from surprise import SVD
svd = SVD(n_factors=100, reg_all=0.02, lr_all=0.005)  # Adjusted hyperparameters
svd.fit(trainset)

from surprise.model_selection import GridSearchCV
param_grid = {
    'n_factors': [50, 100, 150],
    'lr_all': [0.002, 0.005, 0.01],
    'reg_all': [0.02, 0.05, 0.1]
}
grid_search = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=5)
grid_search.fit(data)
print(f"Best RMSE: {grid_search.best_score['rmse']}")
print(f"Best MAE: {grid_search.best_score['mae']}")
print(f"Best Hyperparameters: {grid_search.best_params['rmse']}")

Best RMSE: 1.3197352159869986
Best MAE: 1.2527541094816002
Best Hyperparameters: {'n_factors': 150, 'lr_all': 0.01, 'reg_all': 0.1}
